# CMT — large300 — Nouveaux objectifs de stage (v2)

Reprend le notebook précédent tel quel (setup, objectifs 1/2/4/5) et ajoute
4 pistes discutées à partir des résultats obtenus sur les vraies données :

- **Obj. 2 (suite)** : Lasso + refit post-Lasso pour réduire le nombre de
  termes retenus par STLSQ (NSE=0.617 avec 18 termes -> viser un sous-ensemble
  lisible), et validation walk-forward de ce sous-ensemble (pas de split
  aléatoire sur du temporel).
- **Obj. 4 (suite)** : NSE de reconstruction du flux vs nombre de modes POD,
  et profil de NSE par niveau z (au lieu d'un seul chiffre global à -10.65)
  — pour requalifier l'échec comme confirmation quantifiée de la dominance
  de $T_2$, pas comme un bug.
- **Obj. 5 (suite)** : comparer la position des cellules de circulation
  dominantes au masque PRW humide/sec (lien avec l'auto-agrégation).


## 0. Imports & configuration

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import xarray as xr
import gc, os
from scipy.optimize import curve_fit
from scipy.sparse import diags, kron, identity
from scipy.sparse.linalg import spsolve
from scipy.ndimage import label as ndi_label

plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.25,
    'image.cmap': 'RdBu_r',
})

# ============================================================
#  CONFIGURATION  (identique au notebook rcemip_large300)
# ============================================================
DIR_3D = '3D';  DIR_2D = '2D';  DIR_1D = '1D'
def path3d(var): return os.path.join(DIR_3D, f'MESONH_RCE_large300_3D_{var}.nc')
def path2d(var): return os.path.join(DIR_2D, f'MESONH_RCE_large300_2D_{var}.nc')
def path1d(var): return os.path.join(DIR_1D, f'MESONH_RCE_large300_1D_{var}.nc')

Rd, Rv = 287.05, 461.5
EPSILON = Rd / Rv          # ~ 0.622
BLOC    = 2                # taille de bloc temporel (RAM)

print('Config prête.')


In [ ]:

# --- (0a) Métadonnées : dims, tailles, altitude, fenêtre stationnaire ---
_ds = xr.open_dataset(path3d('ua'));  _da = _ds['ua']
dim_t, dim_z, dim_y, dim_x = _da.dims        # ordre (t, z, y, x)
n_t = _da.sizes[dim_t];  n_z = _da.sizes[dim_z]
n_y = _da.sizes[dim_y];  n_x = _da.sizes[dim_x]
_ds.close();  del _ds, _da;  gc.collect()

_t1 = xr.open_dataset(path1d('ua_avg'))
alt = _t1['altitude'].values.astype(float).copy()
_t1.close()

t_stat   = int(2 * n_t / 3)          # dernier tiers = stationnaire
idx_stat = slice(t_stat, None)
n_stat   = n_t - t_stat
print(f'Grille : {n_t} t x {n_z} z x {n_y} y x {n_x} x')
print(f'Altitude : {alt[0]:.0f} -> {alt[-1]:.0f} m')
print(f'Stationnaire : t={t_stat}->{n_t-1} ({n_stat} pas)')

# grille horizontale (m) — dx/dy = 1000 m par défaut si coords = indices
try:
    xcoord = _ds.coords[dim_x].values.astype(float)
    dx = float(xcoord[1] - xcoord[0])
    if dx < 10:  # coords = indices entiers -> fallback
        raise ValueError
except Exception:
    dx = 1000.0
dy = dx
x = np.arange(n_x) * dx
print(f'dx = dy = {dx:.0f} m')


In [ ]:

# --- (0b) Profil rho_0(z) via température virtuelle (gaz parfaits) ---
rho0 = np.zeros(n_z)
ds_ta  = xr.open_dataset(path3d('ta'))
ds_pa  = xr.open_dataset(path3d('pa'))
ds_hus = xr.open_dataset(path3d('hus'))
for iz in range(n_z):
    ta_z  = ds_ta['ta'].isel({dim_z: iz, dim_t: idx_stat}).values
    pa_z  = ds_pa['pa'].isel({dim_z: iz, dim_t: idx_stat}).values
    hus_z = ds_hus['hus'].isel({dim_z: iz, dim_t: idx_stat}).values
    tv = ta_z * (1.0 + (1.0 / EPSILON - 1.0) * hus_z)
    rho0[iz] = np.mean(pa_z / (Rd * tv))
ds_ta.close(); ds_pa.close(); ds_hus.close(); gc.collect()

fig, ax = plt.subplots(figsize=(4, 5))
ax.plot(rho0, alt / 1000)
ax.set_xlabel(r'$\rho_0$ (kg/m$^3$)'); ax.set_ylabel('z (km)')
ax.set_title(r'Profil $\rho_0(z)$')
fig.tight_layout(); plt.show()


In [ ]:

# --- (0c) Masques humide / sec (PRW), 2D (y,x), seuil = médiane ---
ds_prw = xr.open_dataset(path2d('prw'))
prw_t  = ds_prw['prw'].isel({dim_t: idx_stat}).values     # (n_stat, ny, nx)
ds_prw.close(); gc.collect()
prw = prw_t.mean(axis=0)                                   # moyenne temporelle (ny,nx)
seuil_prw = np.median(prw)
mh = prw > seuil_prw     # humide
ms = ~mh                 # sec
print(f'Seuil PRW = {seuil_prw:.1f} kg/m2  -  {mh.mean()*100:.0f}% de colonnes humides')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.pcolormesh(prw, cmap='YlGnBu', shading='auto')
ax.contour(mh.astype(float), levels=[0.5], colors='k', linewidths=1)
ax.set_title('PRW moyen + contour humide/sec')
fig.colorbar(im, ax=ax, label='PRW (kg/m2)')
fig.tight_layout(); plt.show()


## 1. Visualisation — champs à un temps donné + évolution temporelle

In [ ]:

# --- Profils rho0<u'w'>(z,t), ubar(z,t), wbar(z,t), niveau par niveau ---
flux  = np.zeros((n_z, n_stat))
ubar  = np.zeros((n_z, n_stat))
wbar  = np.zeros((n_z, n_stat))

ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))
for iz in range(n_z):
    u = ds_u['ua'].isel({dim_z: iz, dim_t: idx_stat}).values   # (n_stat, ny, nx)
    w = ds_w['wa'].isel({dim_z: iz, dim_t: idx_stat}).values
    ub = u.mean(axis=(1, 2))
    wb = w.mean(axis=(1, 2))
    up = u - ub[:, None, None]
    wp = w - wb[:, None, None]
    flux[iz, :] = rho0[iz] * (up * wp).mean(axis=(1, 2))
    ubar[iz, :] = ub
    wbar[iz, :] = wb
ds_u.close(); ds_w.close(); gc.collect()

dudz   = np.gradient(ubar, alt, axis=0)
d2udz2 = np.gradient(dudz, alt, axis=0)
print('flux, ubar, wbar, dudz, d2udz2 prêts :', flux.shape)


In [ ]:

def dashboard_at_time(it):
    '''Panneau de profils verticaux a l instant it (indice dans la fenetre stationnaire).'''
    fig, axes = plt.subplots(1, 5, figsize=(17, 5), sharey=True)
    panels = [
        (flux[:, it],   r"$\rho_0\langle u'w'\rangle$"),
        (dudz[:, it],   r"$\partial_z \bar u$"),
        (ubar[:, it],   r"$\bar u$ (m/s)"),
        (wbar[:, it],   r"$\bar w$ (m/s)"),
        (d2udz2[:, it], r"$\partial_z^2 \bar u$"),
    ]
    for ax, (field, label) in zip(axes, panels):
        ax.plot(field, alt / 1000, lw=1.5)
        ax.axvline(0, color='k', lw=0.5)
        ax.set_xlabel(label)
    axes[0].set_ylabel('z (km)')
    fig.suptitle(f'Diagnostics — instant {it} (t={t_stat+it})')
    fig.tight_layout()
    plt.show()

dashboard_at_time(n_stat // 2)


In [ ]:

def dashboard_multi_time(its, cmap_name='viridis'):
    '''Panneau de profils verticaux superposant plusieurs instants (dégradé = temps).'''
    cmap = plt.get_cmap(cmap_name)
    colors = cmap(np.linspace(0, 1, len(its)))

    fig, axes = plt.subplots(1, 5, figsize=(17, 5), sharey=True)
    panels = [
        (flux,   r"$\rho_0\langle u'w'\rangle$"),
        (dudz,   r"$\partial_z \bar u$"),
        (ubar,   r"$\bar u$ (m/s)"),
        (wbar,   r"$\bar w$ (m/s)"),
        (d2udz2, r"$\partial_z^2 \bar u$"),
    ]
    for ax, (field, label) in zip(axes, panels):
        for it, c in zip(its, colors):
            ax.plot(field[:, it], alt / 1000, lw=1.2, color=c)
        ax.axvline(0, color='k', lw=0.5)
        ax.set_xlabel(label)
    axes[0].set_ylabel('z (km)')

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=t_stat, vmax=t_stat + n_stat - 1))
    fig.colorbar(sm, ax=axes, label='indice temporel global', pad=0.01, aspect=30)
    fig.suptitle(f'Diagnostics — {len(its)} instants entre t={t_stat+its[0]} et t={t_stat+its[-1]}')
    plt.show()


its = np.linspace(0, n_stat - 1, 10, dtype=int)
dashboard_multi_time(its)


## 2. Equation discovery — termes physiques grande échelle, restreint à 2-18 km

In [ ]:

# --- restriction verticale, appliquée aux grandeurs déjà calculées sur la
#     grille complète (dudz/d2udz2 restent dérivés sur toute la colonne,
#     puis on tronque — ça évite les artefacts de bord d'une dérivation
#     faite directement sur la fenêtre réduite) ---
mask_z = (alt >= 2000) & (alt <= 18000)
alt_r, rho0_r, ubar_r, wbar_r, dudz_r, d2udz2_r, flux_r = (
    a[mask_z] for a in (alt, rho0, ubar, wbar, dudz, d2udz2, flux)
)
n_z_r = alt_r.size
print(f'{n_z_r} niveaux entre {alt_r[0]:.0f} et {alt_r[-1]:.0f} m (sur {n_z} au total)')

d3udz3_r  = np.gradient(d2udz2, alt, axis=0)[mask_z]
dwdz_r    = np.gradient(wbar, alt, axis=0)[mask_z]
drho0dz_r = np.gradient(rho0, alt)[mask_z][:, None]


def build_library():
    '''Bibliothèque grande échelle, restreinte à 2-18 km.'''
    Z = np.tile(alt_r[:, None], (1, n_stat))
    H = alt.max()                          # sommet convectif = pleine colonne (référence physique)
    rho0_2d = np.tile(rho0_r[:, None], (1, n_stat))

    terms = {
        'dudz':         dudz_r,
        'd2udz2':       d2udz2_r,
        'd3udz3':       d3udz3_r,
        'dudz2':        dudz_r ** 2,
        'absdudz_dudz': np.abs(dudz_r) * dudz_r,
        'dudz_d2udz2':  dudz_r * d2udz2_r,
        'd2udz2_2':     d2udz2_r ** 2,
        'u':        ubar_r,
        'u2':       ubar_r ** 2,
        'u3':       ubar_r ** 3,
        'u_dudz':   ubar_r * dudz_r,
        'u_d2udz2': ubar_r * d2udz2_r,
        'u2_dudz':  ubar_r ** 2 * dudz_r,
        'w':        wbar_r,
        'w2':       wbar_r ** 2,
        'w_dudz':   wbar_r * dudz_r,
        'w_d2udz2': wbar_r * d2udz2_r,
        'dwdz':     dwdz_r,
        'w_u':      wbar_r * ubar_r,
        'rho0_dudz':    rho0_2d * dudz_r,
        'rho0_d2udz2':  rho0_2d * d2udz2_r,
        'drho0dz_u':    drho0dz_r * ubar_r,
        'drho0dz_dudz': drho0dz_r * dudz_r,
        'z_dudz':   Z * dudz_r,
        'z2_dudz':  Z ** 2 * dudz_r,
        'z_d2udz2': Z * d2udz2_r,
        'zHz_dudz': Z * (H - Z) * dudz_r,
        'zHz':      (Z / H) * (1 - Z / H),
    }
    names = list(terms.keys())
    Theta = np.stack([terms[n].ravel() for n in names], axis=1)
    return Theta, names


def nse(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1.0 - ss_res / ss_tot


def stlsq(Theta, y, threshold=0.05, n_iter=15, alpha=1e-8):
    n_terms = Theta.shape[1]
    coef = np.linalg.solve(Theta.T @ Theta + alpha * np.eye(n_terms), Theta.T @ y)
    active = np.ones(n_terms, dtype=bool)
    for _ in range(n_iter):
        small = np.abs(coef) < threshold
        if not np.any(small & active):
            break
        active[small] = False
        if not np.any(active):
            break
        Xs = Theta[:, active]
        coef_active = np.linalg.solve(Xs.T @ Xs + alpha * np.eye(Xs.shape[1]), Xs.T @ y)
        coef = np.zeros(n_terms)
        coef[active] = coef_active
    return coef, active


# --- fit STLSQ ---
Theta, names = build_library()
y = flux_r.ravel()

mu, sigma = Theta.mean(axis=0), Theta.std(axis=0) + 1e-12
Theta_n = (Theta - mu) / sigma
y_n = (y - y.mean()) / (y.std() + 1e-12)

coef_n, active = stlsq(Theta_n, y_n, threshold=0.08)
coef = coef_n * (y.std() / sigma)
intercept = y.mean() - mu @ coef

print('Termes retenus (STLSQ) :')
for name, c, a in zip(names, coef, active):
    if a:
        print(f'  {name:15s} : {c:+.4e}')
print(f'  intercept       : {intercept:+.4e}')

flux_pred = (Theta @ coef + intercept).reshape(n_z_r, n_stat)
r2 = nse(flux_r, flux_pred)
print(f'\nNSE (STLSQ, 2-18 km, z-t complet) = {r2:.3f}')


In [ ]:

# --- contribution en % de chaque terme retenu (poids relatif au profil moyen) ---
importance = {}
for j, (name, c, a) in enumerate(zip(names, coef, active)):
    if not a:
        continue
    contrib = (Theta[:, j] * c).reshape(n_z_r, n_stat).mean(axis=1)
    importance[name] = np.sum(np.abs(contrib))
importance['intercept'] = np.sum(np.abs(np.full(n_z_r, intercept)))

total = sum(importance.values())
pct = {name: 100 * v / total for name, v in importance.items()}
pct_sorted = dict(sorted(pct.items(), key=lambda kv: kv[1], reverse=True))

print('Contribution relative (%) au profil moyen 2-18 km :')
for name, p in pct_sorted.items():
    print(f'  {name:15s} : {p:5.1f}%')

fig, ax = plt.subplots(figsize=(6, 0.4 * len(pct_sorted) + 1))
bars = ax.barh(list(pct_sorted.keys())[::-1], list(pct_sorted.values())[::-1], color='steelblue')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xlabel('contribution relative au profil moyen (%)')
ax.set_title('Poids de chaque terme STLSQ (2-18 km)')
ax.set_xlim(0, max(pct_sorted.values()) * 1.15)
fig.tight_layout(); plt.show()


### 2 bis. Lasso + post-Lasso — réduire le nombre de termes

STLSQ retient beaucoup de termes (souvent redondants entre eux, cf. le bar
chart ci-dessus). Le Lasso force la parcimonie ; le refit OLS sur le support
sélectionné (post-Lasso) lève le biais de rétrécissement du Lasso pur.
$\lambda$ est choisi par **walk-forward CV** (pas de split aléatoire sur du
temporel, cf. le point déjà établi à l'objectif 3).

In [ ]:

def soft_threshold(rho, lam):
    return np.sign(rho) * np.maximum(np.abs(rho) - lam, 0.0)


def lasso_cd(X, y, lam, n_iter=500, tol=1e-7, beta_init=None):
    n_, p_ = X.shape
    beta = np.zeros(p_) if beta_init is None else beta_init.copy()
    r = y - X @ beta
    for _ in range(n_iter):
        b_old = beta.copy()
        for j in range(p_):
            r = r + X[:, j] * beta[j]
            rho = (X[:, j] @ r) / n_
            beta[j] = soft_threshold(rho, lam)
            r = r - X[:, j] * beta[j]
        if np.max(np.abs(beta - b_old)) < tol:
            break
    return beta


def walk_forward_splits(n_samples, n_splits=5):
    fold = n_samples // (n_splits + 1)
    for i in range(1, n_splits + 1):
        tr = np.arange(0, i * fold)
        te = np.arange(i * fold, min((i + 1) * fold, n_samples))
        if len(te) > 0:
            yield tr, te


def walk_forward_splits_blocked(n_stat_, n_z_r_, n_splits=5):
    '''Theta est ravel() avec z qui varie vite -> un fold = des instants t entiers.'''
    for tr_t, te_t in walk_forward_splits(n_stat_, n_splits):
        tr = np.concatenate([np.arange(t * n_z_r_, (t + 1) * n_z_r_) for t in tr_t])
        te = np.concatenate([np.arange(t * n_z_r_, (t + 1) * n_z_r_) for t in te_t])
        yield tr, te


# --- chemin de régularisation ---
lambdas = np.logspace(-3, 0, 25)[::-1]
beta_warm, n_nonzero_path = None, []
for lam in lambdas:
    beta_warm = lasso_cd(Theta_n, y_n, lam, beta_init=beta_warm)
    n_nonzero_path.append(int(np.sum(np.abs(beta_warm) > 1e-8)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(lambdas, n_nonzero_path, 'o-', ms=3)
ax.set_xscale('log'); ax.set_xlabel(r'$\lambda$'); ax.set_ylabel('termes non-nuls')
ax.set_title('Chemin de régularisation Lasso')
fig.tight_layout(); plt.show()

# --- sélection de lambda par walk-forward CV ---
cv_scores = []
for lam in lambdas:
    scores = []
    for tr, te in walk_forward_splits_blocked(n_stat, n_z_r, n_splits=5):
        b = lasso_cd(Theta_n[tr], y_n[tr], lam, n_iter=300)
        pred = Theta_n[te] @ b
        scores.append(nse(y_n[te], pred))
    cv_scores.append(np.mean(scores))

best_i = int(np.argmax(cv_scores))
lam_best = lambdas[best_i]

coef_n_lasso = lasso_cd(Theta_n, y_n, lam_best, n_iter=1000)
if np.sum(np.abs(coef_n_lasso) > 1e-8) == 0:
    print('CV a retenu le modèle vide -> fallback sur le plus petit lambda non-trivial')
    idx_fb = next(i for i, n in enumerate(n_nonzero_path) if n > 0)
    lam_best = lambdas[idx_fb]
    coef_n_lasso = lasso_cd(Theta_n, y_n, lam_best, n_iter=1000)

print(f'lambda retenu = {lam_best:.4f}  (NSE walk-forward CV = {cv_scores[best_i]:.3f})')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(lambdas, cv_scores, 'o-', ms=3)
ax.axvline(lam_best, color='grey', ls='--', lw=0.8)
ax.set_xscale('log'); ax.set_xlabel(r'$\lambda$'); ax.set_ylabel('NSE walk-forward CV')
ax.set_title('Sélection de lambda')
fig.tight_layout(); plt.show()


In [ ]:

# --- post-Lasso : support du Lasso, valeurs par OLS ---
active_lasso = np.abs(coef_n_lasso) > 1e-8
Xa = Theta_n[:, active_lasso]
coef_a, *_ = np.linalg.lstsq(Xa, y_n, rcond=None)
coef_n_final = np.zeros_like(coef_n_lasso)
coef_n_final[active_lasso] = coef_a

coef_final = coef_n_final * (y.std() / sigma)
intercept_final = y.mean() - mu @ coef_final

print(f'{active_lasso.sum()} termes retenus (Lasso), valeurs par OLS :')
for name, cn, c in zip(names, coef_n_final, coef_final):
    if np.abs(cn) > 1e-8:
        print(f'  {name:15s} : {c:+.4e}')
print(f'  intercept       : {intercept_final:+.4e}')

flux_pred_final = (Theta @ coef_final + intercept_final).reshape(n_z_r, n_stat)
r2_final = nse(flux_r, flux_pred_final)
print(f'\nNSE (post-Lasso, {active_lasso.sum()} termes, in-sample) = {r2_final:.3f}')
print(f'(à comparer au NSE STLSQ à {int(active.sum())} termes = {r2:.3f})')

# --- validation walk-forward HONNETE du sous-ensemble retenu (objectif 3) ---
scores_is, scores_oos = [], []
for tr, te in walk_forward_splits_blocked(n_stat, n_z_r, n_splits=5):
    Xa_tr, Xa_te = Theta_n[tr][:, active_lasso], Theta_n[te][:, active_lasso]
    b, *_ = np.linalg.lstsq(Xa_tr, y_n[tr], rcond=None)
    scores_oos.append(nse(y_n[te], Xa_te @ b))
    scores_is.append(nse(y_n[tr], Xa_tr @ b))

print(f'\nValidation walk-forward du sous-ensemble ({active_lasso.sum()} termes) :')
print(f'  NSE moyen en entraînement (in-sample)  = {np.mean(scores_is):+.3f}')
print(f'  NSE moyen en test (walk-forward, hors échantillon) = {np.mean(scores_oos):+.3f}')
print('  (si le 2e chiffre est nettement < le 1er -> overfitting, le sous-ensemble')
print('   ne généralise pas dans le temps malgré le bon NSE in-sample)')

# --- comparaison affichage : vrai vs STLSQ (18 termes) vs post-Lasso (reduit) ---
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot(flux_r.mean(axis=1), alt_r / 1000, color='k', lw=2.2, label='flux vrai')
ax.plot(flux_pred.mean(axis=1), alt_r / 1000, '--', lw=1.4,
        label=f'STLSQ ({int(active.sum())} termes, NSE={r2:.2f})')
ax.plot(flux_pred_final.mean(axis=1), alt_r / 1000, ':', lw=2,
        label=f'post-Lasso ({active_lasso.sum()} termes, NSE={r2_final:.2f})')
ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel(r"$\rho_0\langle u'w'\rangle$"); ax.set_ylabel('z (km)')
ax.legend(); ax.set_title('Profil moyen 2-18 km : vrai vs régressions')
fig.tight_layout(); plt.show()


## 4. $\psi$ moyennée en $y$ → POD

In [ ]:

def psi_poisson_periodic(U2d, W2d, xv, zv, rho0v):
    '''nabla^2 psi = d_x(rho0 w) - d_z(rho0 u), periodique en x, Dirichlet en z.'''
    nz, nx = U2d.shape
    dxl = xv[1] - xv[0]
    rW = rho0v[:, None] * W2d
    rU = rho0v[:, None] * U2d
    dxrW = (np.roll(rW, -1, axis=1) - np.roll(rW, 1, axis=1)) / (2 * dxl)
    dzrU = np.gradient(rU, zv, axis=0)
    omega = dxrW - dzrU

    main_x = -2 * np.ones(nx); off_x = np.ones(nx - 1)
    Lx = diags([main_x, off_x, off_x, [1], [1]], [0, 1, -1, nx - 1, -(nx - 1)]) / dxl ** 2
    dz_mean = np.mean(np.diff(zv))
    main_z = -2 * np.ones(nz); off_z = np.ones(nz - 1)
    Lz = diags([main_z, off_z, off_z], [0, 1, -1]) / dz_mean ** 2
    A = (kron(identity(nz), Lx) + kron(Lz, identity(nx))).tolil()
    b = omega.ravel().copy()

    psi_top = np.cumsum(rho0v * U2d.mean(axis=1)) * dz_mean
    for i in range(nx):
        k0 = i;               A.rows[k0] = [k0]; A.data[k0] = [1.0]; b[k0] = 0.0
        k1 = (nz - 1) * nx + i; A.rows[k1] = [k1]; A.data[k1] = [1.0]; b[k1] = psi_top[-1]
    psi = spsolve(A.tocsr(), b).reshape(nz, nx)
    return psi


In [ ]:

# --- construction des snapshots psi(x,z,t) moyenne-y, sous-echantillonnage temporel raisonnable ---
STRIDE_T = max(1, n_stat // 60)     # ~60 instants au plus pour rester rapide
t_idx_pod = np.arange(0, n_stat, STRIDE_T)

psi_snaps = np.zeros((len(t_idx_pod), n_z, n_x))
ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))
for i_snap, it in enumerate(t_idx_pod):
    it_global = t_stat + it
    U2d = ds_u['ua'].isel({dim_t: it_global}).mean(dim=dim_y).values   # (nz, nx)
    W2d = ds_w['wa'].isel({dim_t: it_global}).mean(dim=dim_y).values
    psi_snaps[i_snap] = psi_poisson_periodic(U2d, W2d, x, alt, rho0)
ds_u.close(); ds_w.close(); gc.collect()
print('psi_snaps :', psi_snaps.shape)


In [ ]:

# --- POD methode des snapshots (centree !) ---
N = psi_snaps.shape[0]
fmean = psi_snaps.mean(axis=0)
Amat = (psi_snaps - fmean).reshape(N, -1)          # ANOMALIES centrees

Cs = Amat @ Amat.T / N
evals, evecs = np.linalg.eigh(Cs)
order = evals.argsort()[::-1]
evals = np.clip(evals[order], 0, None)
evecs = evecs[:, order]

modes_flat = evecs.T @ Amat                          # (N, nz*nx)
norms = np.linalg.norm(modes_flat, axis=1) + 1e-30
modes_flat = modes_flat / norms[:, None]
modes = modes_flat.reshape(N, n_z, n_x)
a_n = Amat @ modes_flat.T                             # (N_t, N_modes) coefficients temporels

energy = evals / evals.sum()
cum_energy = np.cumsum(energy)
n_modes = int(np.searchsorted(cum_energy, 0.90) + 1)
print(f'{n_modes} modes pour 90% de l energie (sur {len(cum_energy)} modes).')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, len(cum_energy) + 1), cum_energy, 'o-', ms=3)
ax.axhline(0.90, color='grey', ls='--', lw=0.8)
ax.axvline(n_modes, color='grey', ls='--', lw=0.8)
ax.set_xlabel('nombre de modes'); ax.set_ylabel('énergie cumulée')
ax.set_title('Critère de troncature POD')
fig.tight_layout(); plt.show()


In [ ]:

def reconstruct_uw(n1, n2):
    '''u~_N = d(psi_N)/dz / rho0,  w~_N = -d(psi_N)/dx / rho0, pour les modes n1..n2.'''
    psi_N = fmean[None, :, :] + np.einsum('tn,nzx->tzx', a_n[:, n1:n2 + 1], modes[n1:n2 + 1])
    u_N = np.gradient(psi_N, alt, axis=1) / rho0[None, :, None]
    w_N = -np.gradient(psi_N, x, axis=2) / rho0[None, :, None]
    return psi_N, u_N, w_N


psi_N, u_N, w_N = reconstruct_uw(0, n_modes - 1)

up_N = u_N - u_N.mean(axis=(0, 2), keepdims=True)
wp_N = w_N - w_N.mean(axis=(0, 2), keepdims=True)
flux_N_profile = np.mean(up_N * wp_N, axis=(0, 2)) * rho0

flux_true_profile = flux[:, t_idx_pod].mean(axis=1)
nse_recon = nse(flux_true_profile, flux_N_profile)
print(f'NSE reconstruction flux (via {n_modes} modes POD) = {nse_recon:.3f}')
print('(rappel : psi est moyenne-y, donc cette reconstruction ne capture que la')
print(' partie organisee T1 — coherent avec le residu T2 deja identifie ailleurs.)')

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(flux_true_profile, alt / 1000, label='flux complet (3D)')
ax.plot(flux_N_profile, alt / 1000, '--', label=f'{n_modes} modes POD')
ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel(r"$\rho_0\langle u'w'\rangle$"); ax.set_ylabel('z (km)')
ax.legend(); ax.set_title('Reconstruction POD du flux')
fig.tight_layout(); plt.show()


### 4 bis. Requalifier l'échec : NSE vs nombre de modes, et profil par niveau

Un seul NSE global négatif ne dit pas *où* ni *pourquoi* ça échoue. On regarde :
1. si le NSE s'améliore en ajoutant plus de modes que les {n_modes} à 90% d'énergie
   (si non -> limite physique, $T_2$ domine partout ; si oui -> les modes
   d'énergie faible portent l'info utile, il faudrait un critère de sélection
   basé sur la corrélation au flux plutôt que sur l'énergie de $\psi$) ;
2. un profil de NSE **par niveau z**, pour voir si $T_1$ explique quelque chose
   à certaines altitudes même si l'agrégat est catastrophique.

In [ ]:

def nse_profile_by_level(y_true_zt, y_pred_zt):
    '''NSE calcule independamment a chaque niveau z (chaque ligne).'''
    nse_z = np.full(y_true_zt.shape[0], np.nan)
    for iz in range(y_true_zt.shape[0]):
        yt, yp = y_true_zt[iz], y_pred_zt[iz]
        ss_tot = np.sum((yt - yt.mean()) ** 2)
        if ss_tot > 1e-30:
            nse_z[iz] = 1.0 - np.sum((yt - yp) ** 2) / ss_tot
    return nse_z


# --- NSE global vs nombre de modes ---
n_modes_max = min(30, modes.shape[0])
nse_vs_nmodes = []
for n in range(1, n_modes_max + 1):
    _, u_n, w_n = reconstruct_uw(0, n - 1)
    up = u_n - u_n.mean(axis=(0, 2), keepdims=True)
    wp = w_n - w_n.mean(axis=(0, 2), keepdims=True)
    flux_n_profile = np.mean(up * wp, axis=(0, 2)) * rho0
    nse_vs_nmodes.append(nse(flux_true_profile, flux_n_profile))

fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.plot(range(1, n_modes_max + 1), nse_vs_nmodes, 'o-', color='C0', label='NSE flux reconstruit')
ax1.set_xlabel('nombre de modes'); ax1.set_ylabel('NSE (flux)', color='C0')
ax1.axhline(0, color='k', lw=0.5)
ax1.axvline(n_modes, color='grey', ls='--', lw=0.8)

ax2 = ax1.twinx()
ax2.plot(range(1, n_modes_max + 1), cum_energy[:n_modes_max], 's--', color='C1', alpha=0.6,
         label=r'énergie cumulée $\psi$')
ax2.set_ylabel('énergie cumulée', color='C1')
fig.suptitle('Energie de psi vs fidélité de reconstruction du flux')
fig.tight_layout(); plt.show()

print(f'NSE a {n_modes} modes (90% energie) : {nse_vs_nmodes[n_modes-1]:.3f}')
print(f'NSE a {n_modes_max} modes (max testé) : {nse_vs_nmodes[-1]:.3f}')


In [ ]:

# --- profil de NSE par niveau, a n_modes fixe (perturbation par rapport a la
#     moyenne en x seulement, temps garde -> comparable a flux[:, t_idx_pod]) ---
up_N_txz = u_N - u_N.mean(axis=2, keepdims=True)
wp_N_txz = w_N - w_N.mean(axis=2, keepdims=True)
flux_N_txz = rho0[None, :, None] * up_N_txz * wp_N_txz     # (n_snap, nz, nx)
flux_N_zt = flux_N_txz.mean(axis=2).T                       # (nz, n_snap)

flux_true_zt = flux[:, t_idx_pod]                            # (nz, n_snap), memes instants

nse_z = nse_profile_by_level(flux_true_zt, flux_N_zt)

fig, ax = plt.subplots(figsize=(5, 6))
ax.plot(nse_z, alt / 1000, 'o-', ms=3)
ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel('NSE (par niveau)'); ax.set_ylabel('z (km)')
ax.set_title(f'NSE reconstruction flux par niveau ({n_modes} modes)')
fig.tight_layout(); plt.show()

n_positive = np.sum(nse_z > 0)
print(f'{n_positive}/{len(nse_z)} niveaux ont un NSE positif (T1 explique quelque chose)')
if n_positive > 0:
    iz_best = np.nanargmax(nse_z)
    print(f'Meilleur niveau : z={alt[iz_best]/1000:.1f} km, NSE={nse_z[iz_best]:.3f}')


## 5. Circulation — cellules fermées de $\psi$, et lien avec le masque PRW

In [ ]:

def vorticity_uw(U2d, W2d, xv, zv):
    '''Vorticite physique omega_y = d(u)/dz - d(w)/dx (Stokes), differente
    du second membre du solveur de Poisson.'''
    dudz = np.gradient(U2d, zv, axis=0)
    dwdx = np.gradient(W2d, xv, axis=1)
    return dudz - dwdx


def circulation_cells(psi_a, omega, dxl, dzl, min_size=25):
    '''Segmente psi_a par signe constant (composantes connexes), calcule
    Gamma = somme(omega)*dx*dz par cellule (Stokes). min_size filtre le
    bruit de resolution du solveur.'''
    sign_map = np.where(psi_a >= 0, 1, -1)
    cells = []
    for s in (1, -1):
        mask = sign_map == s
        lbl, n_comp = ndi_label(mask)
        for k in range(1, n_comp + 1):
            comp_mask = lbl == k
            if comp_mask.sum() < min_size:
                continue
            coords = np.argwhere(comp_mask)
            gamma = omega[comp_mask].sum() * dxl * dzl
            cz, cx = coords.mean(axis=0)
            cells.append(dict(sign=s, area=int(comp_mask.sum()), gamma=gamma,
                               centroid=(cz, cx), coords=coords))
    cells.sort(key=lambda c: -abs(c['gamma']))
    return cells, sign_map


it_mid = t_idx_pod[len(t_idx_pod) // 2]
psi_mid = psi_snaps[len(t_idx_pod) // 2]
psi_a_mid = psi_mid - psi_mid.mean()

ds_u = xr.open_dataset(path3d('ua'))
ds_w = xr.open_dataset(path3d('wa'))
U2d_mid = ds_u['ua'].isel({dim_t: t_stat + it_mid}).mean(dim=dim_y).values
W2d_mid = ds_w['wa'].isel({dim_t: t_stat + it_mid}).mean(dim=dim_y).values
ds_u.close(); ds_w.close(); gc.collect()

omega_mid = vorticity_uw(U2d_mid, W2d_mid, x, alt)
dz_mean = np.mean(np.diff(alt))

cells, sign_map = circulation_cells(psi_a_mid, omega_mid, dx, dz_mean, min_size=25)

print(f'{len(cells)} cellules de circulation significatives :')
for i, c in enumerate(cells[:10]):
    cz, cx = c['centroid']
    sens = 'horaire' if c['gamma'] < 0 else 'anti-horaire'
    print(f"  #{i:2d}  signe={c['sign']:+d}  Gamma={c['gamma']:+.2e}  ({sens})  "
          f"aire={c['area']:4d} pts  centre=(z={alt[int(cz)]/1000:.1f}km, x={x[int(cx)]/1000:.0f}km)")

cell_map = np.full(psi_a_mid.shape, np.nan)
for i, c in enumerate(cells):
    for (zz, xx) in c['coords']:
        cell_map[zz, xx] = i

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
vmax = np.nanpercentile(np.abs(psi_a_mid), 98) + 1e-30
axes[0].pcolormesh(x / 1000, alt / 1000, psi_a_mid, cmap='RdBu_r',
                    vmin=-vmax, vmax=vmax, shading='auto')
axes[0].set_title(r"$\psi'$ (relief)")

cmap_cells = plt.cm.get_cmap('tab20', max(len(cells), 1))
axes[1].pcolormesh(x / 1000, alt / 1000, cell_map, cmap=cmap_cells, shading='auto')
for i, c in enumerate(cells):
    cz, cx = c['centroid']
    axes[1].annotate(str(i), (x[int(cx)] / 1000, alt[int(cz)] / 1000),
                      fontsize=8, ha='center', va='center', fontweight='bold')
axes[1].set_title(f'{len(cells)} cellules de circulation (numérotées)')

for ax in axes:
    ax.set_xlabel('x (km)'); ax.set_ylabel('z (km)')
fig.tight_layout(); plt.show()


### 5 bis. Les cellules dominantes coïncident-elles avec le masque PRW ?

Si la bande ascendante correspond aux colonnes humides (`mh`) et la bande
descendante aux colonnes sèches (`ms`), ça relie directement la circulation
moyenne-y à l'auto-agrégation convective (déjà dans tes lectures : Wing et al.).

In [ ]:

# --- profil PRW moyenné en y, en fonction de x ---
prw_x = prw.mean(axis=0)   # (nx,)

# --- pour chaque cellule dominante (les 2 plus grosses par aire), l'etendue en x
#     et la valeur moyenne de PRW sur cette etendue ---
cells_by_area = sorted(cells, key=lambda c: -c['area'])[:2]

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                          gridspec_kw={'height_ratios': [2, 1]})

vmax = np.nanpercentile(np.abs(psi_a_mid), 98) + 1e-30
axes[0].pcolormesh(x / 1000, alt / 1000, psi_a_mid, cmap='RdBu_r',
                    vmin=-vmax, vmax=vmax, shading='auto')
axes[0].set_ylabel('z (km)')
axes[0].set_title(r"$\psi'$ et étendue en x des cellules dominantes")

axes[1].plot(x / 1000, prw_x, color='k', lw=1.2)
axes[1].axhline(seuil_prw, color='grey', ls='--', lw=1, label='seuil PRW (médiane)')
axes[1].set_xlabel('x (km)'); axes[1].set_ylabel('PRW moy-y (kg/m2)')

colors_cell = ['tab:red', 'tab:blue']
print('Cellules dominantes vs PRW :')
for c, col in zip(cells_by_area, colors_cell):
    ix = c['coords'][:, 1]
    ix_min, ix_max = ix.min(), ix.max()
    prw_cell = prw_x[ix_min:ix_max + 1].mean()
    label_humid = 'humide' if prw_cell > seuil_prw else 'sec'
    sens = 'ascendant (anti-horaire)' if c['gamma'] > 0 else 'descendant (horaire)'
    print(f"  cellule signe={c['sign']:+d} ({sens}) : x=[{x[ix_min]/1000:.0f},"
          f"{x[ix_max]/1000:.0f}]km, PRW moyen={prw_cell:.1f} kg/m2 -> {label_humid} "
          f"(seuil={seuil_prw:.1f})")

    for ax in axes:
        ax.axvspan(x[ix_min] / 1000, x[ix_max] / 1000, color=col, alpha=0.15)
    axes[1].axvspan(x[ix_min] / 1000, x[ix_max] / 1000, color=col, alpha=0.15,
                     label=f"cellule signe={c['sign']:+d}")

axes[1].legend(fontsize=8)
fig.tight_layout(); plt.show()


## Synthèse

- **Obj. 2** : le post-Lasso donne une fermeture avec beaucoup moins de termes ;
  la validation walk-forward dit si elle généralise dans le temps ou si le
  bon NSE in-sample était trompeur.
- **Obj. 4** : la courbe NSE-vs-modes et le profil par niveau distinguent une
  limite physique ($T_2$ domine partout) d'un problème de critère de sélection
  des modes (l'énergie de $\psi$ n'est pas le bon critère si le NSE continue
  à grimper au-delà du coude à 90%).
- **Obj. 5** : si les cellules dominantes s'alignent avec `mh`/`ms`, la
  circulation moyenne-y est le signal d'auto-agrégation à l'échelle du
  domaine, pas une structure convective locale — un résultat physique en soi,
  pas un échec de la méthode de segmentation.
